# Zipformer CS (VI+EN) — Data Processing

**CPU runtime.** Builds the training data and pushes heavy artifacts to the Hugging Face Hub so the train notebook can restore them:
**setup → select VI+EN → materialize → fbank → MUSAN → push.**

Shared config lives in `colab/colab_config.yaml` (edited once, used by both notebooks). Set `force_recompute: true` there to skip all Hub pulls, recompute fbank + MUSAN from scratch, and push the fresh artifacts.

## 0. Setup

In [ ]:
# Python deps for data processing (CPU). PyYAML is preinstalled on Colab but
# pinned here for non-Colab runtimes.
!pip -q install lhotse datasets soundfile sentencepiece wordfreq kaldialign \
  num2words huggingface_hub pyyaml
import lhotse, datasets
print('lhotse:', lhotse.__version__, '| datasets:', datasets.__version__)

In [ ]:
# --- Load shared config -----------------------------------------------------
# Single edit point for BOTH notebooks: colab/colab_config.yaml.
import os, yaml

CONFIG_PATH = 'colab/colab_config.yaml'   # relative to the repo root
with open(CONFIG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

REPO       = cfg['repo']
DRIVE      = cfg['drive']
PREFIX     = cfg['prefix']
BPE_MODEL  = cfg['bpe_model']
PRETRAINED = cfg['pretrained']
TEXT_NORM  = cfg['text_norm']
HF_DATA_REPO  = cfg['hf_data_repo']
HF_MODEL_REPO = cfg['hf_model_repo']
FORCE_RECOMPUTE = bool(cfg['force_recompute'])

# Derived paths (kept identical across both notebooks).
ASR      = f'{REPO}/icefall/egs/librispeech/ASR'
FBANK    = 'data/fbank'
EXP_DIR  = 'data/exp_finetune'
COMBINED = 'data/combined_dataset'
MANIFEST = 'data/combined_manifest.jsonl'
HF_SYNC  = os.path.abspath(f'{REPO}/colab/hf_sync.py')  # absolute: survives %cd

os.makedirs(DRIVE, exist_ok=True)
assert os.path.isdir(REPO), f'Put this repo at {REPO} (clone or upload).'
print('REPO =', REPO, '| DRIVE =', DRIVE, '| FORCE_RECOMPUTE =', FORCE_RECOMPUTE)

In [ ]:
# Place the repo's maintained scripts where the recipe expects them.
import shutil
shutil.copy(f'{REPO}/compute_fbank_huggingface.py', f'{ASR}/local/compute_fbank_huggingface.py')
print('compute_fbank_huggingface.py placed under', ASR)

## 0b. Hugging Face Hub (persistence)

Log in once per session, then restore/push artifacts. Leave the `hf_*_repo` config keys empty to skip the Hub and use local paths only.

In [ ]:
# Auth once per session (or set os.environ['HF_TOKEN'] = '...').
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Inputs (bpe.model + base ckpt): push once, or restore after a restart.
# FORCE_RECOMPUTE skips the restore pull (still pushes a local copy).
if HF_MODEL_REPO:
    if os.path.exists(PRETRAINED):
        !python {HF_SYNC} push --repo-id {HF_MODEL_REPO} --repo-type model --private --local {BPE_MODEL}
        !python {HF_SYNC} push --repo-id {HF_MODEL_REPO} --repo-type model --private --local {PRETRAINED}
    elif not FORCE_RECOMPUTE:
        !python {HF_SYNC} pull --repo-id {HF_MODEL_REPO} --repo-type model --local model \
          --allow-pattern 'bpe.model' --allow-pattern 'epoch-35-avg-*.pt'
    else:
        print('FORCE_RECOMPUTE - skipping input restore; upload bpe/ckpt manually.')
else:
    print('hf_model_repo empty - skipping input push/restore.')

## 1. Filter sanity check

In [ ]:
%cd {REPO}/colab
!python test_codeswitch_filter.py

## 2. Select VI+EN utterances

Edit `sources` with your candidate datasets. `text_key`/`audio_key` are the column names in each dataset. Start with `--limit` for a dry run, review, tune, then drop `--limit` for the full scan.

In [ ]:
import json
sources = [
    {'dataset': 'org/your-vi-dataset-1', 'config': None, 'split': 'train',
     'text_key': 'transcription', 'audio_key': 'audio'},
    # {'dataset': 'org/your-vi-dataset-2', 'split': 'train', 'text_key': 'sentence'},
]
with open('sources.json', 'w', encoding='utf-8') as f:
    json.dump(sources, f, ensure_ascii=False, indent=2)
print(open('sources.json', encoding='utf-8').read())

In [ ]:
# Dry run: scan up to 5000 rows/source. Remove --limit for the real selection.
!python select_codeswitch_utterances.py \
  --sources-json sources.json \
  --output-dir {DRIVE} \
  --min-en-tokens 1 \
  --limit 5000

In [ ]:
# Review precision before scaling. Tune --min-en-tokens / --max-en-ratio /
# wordlists if too many false positives appear here.
print('=== SELECTED (english tokens shown) ===')
print(open(f'{DRIVE}/samples_selected.txt', encoding='utf-8').read()[:4000])
print('\n=== REJECTED ===')
print(open(f'{DRIVE}/samples_rejected.txt', encoding='utf-8').read()[:2000])
print('\n=== STATS ===')
print(open(f'{DRIVE}/selection_stats.json', encoding='utf-8').read())

## 3. BPE coverage check (BLOCKING)

Confirm your BPE tokenizes the selected (normalized) transcripts without an `<unk>` flood and that `text_norm` round-trips Vietnamese correctly. If this looks wrong, fix `text_norm` (likely `'none'`) before computing features.

In [ ]:
import json, sentencepiece as spm, sys
sys.path.insert(0, f'{ASR}/local')
from compute_fbank_huggingface import normalize_text

sp = spm.SentencePieceProcessor(); sp.load(BPE_MODEL)
unk = sp.piece_to_id('<unk>')
rows = [json.loads(l) for l in open(MANIFEST, encoding='utf-8')][:20]
tot = unkn = 0
for r in rows:
    norm = normalize_text(r['text'], TEXT_NORM)
    ids = sp.encode(norm, out_type=int)
    tot += len(ids); unkn += sum(i == unk for i in ids)
    print(f"{norm}\n  -> {sp.encode(norm, out_type=str)}\n")
print(f'tokens={tot} unk={unkn} ({100*unkn/max(tot,1):.1f}%)')
assert tot == 0 or unkn / tot < 0.02, 'Too many <unk>; check text_norm / BPE.'

## 4. Materialize combined dataset to Drive

In [ ]:
!python materialize_dataset.py \
  --manifest {MANIFEST} \
  --output-dir {COMBINED} \
  --sampling-rate 16000 \
  --dev-frac 0.02 \
  --test-frac 0.02

## 5. Compute fbank (train perturbed, dev clean) → Drive

With `force_recompute: false`, a fresh session pulls the dataset + fbank (incl. MUSAN) from the Hub to skip recompute. With `force_recompute: true`, the pull is skipped and everything is recomputed locally.

In [ ]:
%cd {ASR}
import os
# Recompute if forced, or if no local cuts yet. Pull from the Hub only when
# NOT forcing (force == build fresh, ignore Hub state).
need_fbank = FORCE_RECOMPUTE or not os.path.exists(f'{FBANK}/{PREFIX}_cuts_train.jsonl.gz')
if HF_DATA_REPO and need_fbank and not FORCE_RECOMPUTE:
    !python {HF_SYNC} pull --repo-id {HF_DATA_REPO} --repo-type dataset --local data --path-in-repo combined_dataset
    !python {HF_SYNC} pull --repo-id {HF_DATA_REPO} --repo-type dataset --local data --path-in-repo fbank
    need_fbank = not os.path.exists(f'{FBANK}/{PREFIX}_cuts_train.jsonl.gz')
if need_fbank:
    print('RUN the compute cell below.' + (' (FORCE_RECOMPUTE)' if FORCE_RECOMPUTE else ''))
else:
    print('Fbank cuts present - you may SKIP the compute cell below.')

In [ ]:
%cd {ASR}
import os, json
os.makedirs(FBANK, exist_ok=True)

# 'train' gets 3x speed perturbation; 'dev'/'test' stay clean. If 'test' is
# missing, re-run the materialize cell with --test-frac > 0, then re-run this.
with open(f'{COMBINED}/dataset_dict.json') as f:
    splits = json.load(f)['splits']
print('combined_dataset splits:', splits)

for split in splits:
    perturb = '--perturb-speed' if split == 'train' else ''
    !python local/compute_fbank_huggingface.py \
      --load-from-disk --dataset {COMBINED} --split {split} \
      --output-name {split} --prefix {PREFIX} --output-dir {FBANK} \
      --text-normalization {TEXT_NORM} --resample-rate 16000 \
      --bpe-model {BPE_MODEL} {perturb}

In [ ]:
# Persist dataset + fbank cuts to the Hub (run after fbank computes).
if HF_DATA_REPO:
    !python {HF_SYNC} push --repo-id {HF_DATA_REPO} --repo-type dataset --private --local {COMBINED} --path-in-repo combined_dataset
    # whole fbank dir: cuts (*.jsonl.gz) AND the lilcom *_feats_* features.
    !python {HF_SYNC} push --repo-id {HF_DATA_REPO} --repo-type dataset --private --local {FBANK} --path-in-repo fbank
else:
    print('hf_data_repo empty - skipping dataset/fbank push.')

In [ ]:
# Duration distribution sanity check.
!python local/display_manifest_statistics.py 2>/dev/null || \
  python -c "from lhotse import load_manifest_lazy; \
cs=load_manifest_lazy('{FBANK}/{PREFIX}_cuts_train.jsonl.gz'); \
import itertools; print('train cuts:', len(list(itertools.islice((c for c in cs),0,10**9))))"

## 5b. MUSAN noise prep (CPU)

Download MUSAN, compute its fbank (80 mel bins / 16 kHz, matching train), build a **speech-only** subset, then a **noisy copy** of the held-out test set. All artifacts land in `{FBANK}` and are pushed below, so a later session restores them instead of re-downloading the ~11 GB raw corpus.

In [ ]:
%cd {ASR}
import os
# MUSAN fbank (matches train: 80 mel bins, 16 kHz). Recompute if forced, else
# skip when already present (e.g. pulled from the Hub). Raw corpus ~11 GB.
MUSAN_DL = 'data/musan_dl'  # raw download dir (not persisted)
if FORCE_RECOMPUTE or not os.path.exists(f'{FBANK}/musan_cuts.jsonl.gz'):
    if not os.path.isdir(f'{MUSAN_DL}/musan'):
        !lhotse download musan {MUSAN_DL}
    !lhotse prepare musan {MUSAN_DL}/musan data/manifests
    !python local/compute_fbank_musan.py --output-dir {FBANK}
else:
    print('musan_cuts.jsonl.gz present - skipping MUSAN compute.')

In [ ]:
%cd {ASR}
# Speech-only MUSAN manifest (human background) for MUSAN_SOURCE='speech'.
from lhotse import load_manifest
musan = load_manifest(f'{FBANK}/musan_cuts.jsonl.gz')
speech = musan.filter(lambda c: (c.recording_id or '').startswith('speech')).to_eager()
out = f'{FBANK}/musan_speech_cuts.jsonl.gz'
speech.to_file(out)
print(f'speech cuts: {len(speech)} -> {out}')

In [ ]:
%cd {ASR}
# Build a noisy copy of the held-out test set (reuses the training CutMix).
MK_NOISY = HF_SYNC.replace('hf_sync.py', 'make_noisy_cuts.py')
!python {MK_NOISY}   --test-cuts {FBANK}/{PREFIX}_cuts_test.jsonl.gz   --musan-cuts {FBANK}/musan_cuts.jsonl.gz   --snr-low 5 --snr-high 20   --output {FBANK}/{PREFIX}_cuts_test_noisy.jsonl.gz

In [ ]:
# Persist MUSAN + noisy artifacts to the Hub so future sessions skip the
# ~11 GB download AND the fbank compute. upload_folder dedups the already-
# pushed train/dev/test feats, so effectively only musan_cuts /
# musan_speech_cuts / musan_feats* / {PREFIX}_cuts_test_noisy upload here.
if HF_DATA_REPO:
    !python {HF_SYNC} push --repo-id {HF_DATA_REPO} --repo-type dataset --private \
      --local {FBANK} --path-in-repo fbank
else:
    print('hf_data_repo empty - skipping MUSAN/noisy push.')